# neuro_rave quickstart

This notebook fits **RAVE** and **RAVE+** on synthetic time-series data recorded in three sessions ("domains"). It shows how to:

1. shape your data for `neuro_rave`,
2. learn an embedding meant to suppress session information with `RAVE` (no class labels needed),
3. reconstruct the time series and compare them with the inputs,
4. train `RAVEPlus` with **partially known class labels** (`None` = unknown) and predict classes for an unlabeled session.

It runs on CPU in under a minute. The method comes from Gonschorek et al., *Removing Inter-Experimental Variability from Functional Data in Systems Neuroscience*, NeurIPS 2021 ([paper](https://proceedings.neurips.cc/paper_files/paper/2021/hash/1e5eeb40a3fce716b244599862fd2200-Abstract.html), [original code](https://github.com/eulerlab/rave)).

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np

import neuro_rave
from neuro_rave import RAVE, RAVEPlus

print("python    :", sys.executable)
print("neuro_rave:", neuro_rave.__version__, "from", neuro_rave.__file__)

## 1. Input shapes

`neuro_rave` takes NumPy arrays, and each row is one fixed-length time series:

| Argument | Shape | Content |
|---|---|---|
| `X` | `(n_samples, time)` or `(n_samples, time, channels)` | Real-valued traces on a common time grid and channel order |
| `domains` | `(n_samples,)` | One domain ID per row (for example a session or experiment), as strings or integers, with at least two domains |
| `y` (`RAVEPlus` only) | `(n_samples,)` | One class label per row, **`None` where unknown**, with at least two observed classes |

Cutting recordings to a common length, resampling, alignment and scaling are up to you; the package does none of them.

Here we simulate three waveform classes (`transient`, `sustained`, `oscillating`) in two channels, recorded in sessions `A`, `B` and `C`. Each session distorts the time series with its own gain, offset, small time shift and slow background component. Class labels are kept for sessions `A` and `B` and hidden (`None`) for session `C`.

In [ ]:
T, CHANNELS = 60, 2
CLASSES = np.array(["transient", "sustained", "oscillating"])
SESSIONS = np.array(["A", "B", "C"])
t = np.linspace(0.0, 1.0, T)
TEMPLATES = {
    "transient": np.exp(-(((t - 0.25) / 0.05) ** 2)),
    "sustained": 1 / (1 + np.exp(-(t - 0.2) / 0.02)) - 1 / (1 + np.exp(-(t - 0.7) / 0.02)),
    "oscillating": np.sin(2 * np.pi * 4 * t) * np.exp(-(((t - 0.5) / 0.25) ** 2)),
}
# gain, offset, time shift (samples), phase of the slow background component
SESSION_EFFECTS = {"A": (1.0, 0.0, 0, 0.0), "B": (1.4, 0.3, 2, 2.0), "C": (0.7, -0.3, -2, 4.0)}


def simulate(n_per_session, rng):
    series, sessions, classes = [], [], []
    for session in SESSIONS:
        gain, offset, shift, phase = SESSION_EFFECTS[session]
        slow = 0.6 * np.cos(2 * np.pi * t + phase)
        for label in rng.choice(CLASSES, size=n_per_session):
            base = TEMPLATES[label] * rng.uniform(0.8, 1.2)
            channels = np.stack([base, 0.6 * np.roll(base, 3)], axis=-1)
            distorted = gain * np.roll(channels, shift, axis=0) + offset + slow[:, None]
            series.append(distorted + 0.15 * rng.standard_normal((T, CHANNELS)))
            sessions.append(session)
            classes.append(label)
    return np.asarray(series), np.asarray(sessions), np.asarray(classes)


rng = np.random.default_rng(0)
X, domains, true_classes = simulate(300, rng)  # training time series
X_new, domains_new, y_new_true = simulate(100, rng)  # new time series from the same sessions

# Scaling is the caller's job: standardize each (time, channel) with training statistics.
mean, std = X.mean(axis=0), X.std(axis=0) + 1e-8
X, X_new = (X - mean) / std, (X_new - mean) / std

# Partially known labels for RAVEPlus: None marks an unknown label (here: all of session C).
y = [None if d == "C" else c for d, c in zip(domains, true_classes, strict=True)]

print("X:", X.shape, "| domains:", sorted(set(domains.tolist())))
print("known labels:", sum(v is not None for v in y), "of", len(y))

### Using your own data

Replace the cell above with your own arrays under the same names. Everything below uses only these names, whatever your time-series length, channel count, domain IDs or class labels:

| Name | Needed for | Content |
|---|---|---|
| `X`, `domains` | fitting | training time series and one domain ID per row |
| `y` | `RAVEPlus` | one label per training row, `None` where unknown |
| `X_new` | inference | new time series with the same `(time[, channels])` layout as `X` (can simply be `X`) |
| `domains_new` | plots, scores | domain ID of each row of `X_new` |
| `y_new_true` | scores (optional) | true labels of `X_new` if you have them, otherwise `None` |

For example:

```python
X = np.load("my_series.npy")                               # (n_samples, time) or (n_samples, time, channels)
domains = np.load("my_sessions.npy", allow_pickle=True)    # one ID per row
y = [label if known else None for label, known in zip(labels, is_labeled)]
X_new, domains_new, y_new_true = X, domains, None
```

## 2. RAVE: suppressing session information without class labels

`RAVE` trains an encoder and decoder against an adversarial session discriminator, using only `X` and `domains`. This notebook sets `domain_weight=1.0` explicitly, the value its results were checked with on this synthetic data. The package default is `0.1`, chosen from a comparison on recorded data (see the README); start from the default on your own data.

In [ ]:
# domain_weight=1.0 is the setting this notebook's outputs were verified with on the
# synthetic data; the package default is 0.1 (see the README).
rave = RAVE(latent_dim=8, domain_weight=1.0, seed=0).fit(X, domains=domains)

Z = rave.transform(X_new)  # (n_samples, latent_dim)
X_rec = rave.reconstruct(X_new)  # same shape as X_new
print("embedding:", Z.shape, "| reconstruction:", X_rec.shape)

## 3. Inputs vs. reconstructions

The reconstruction is decoded from the embedding, and the adversary pushes the embedding to drop whatever only identifies the session. Reconstructions are therefore not expected to be perfect. The plot shows the first new time series of each domain.

In [ ]:
inputs, recons = (
    np.atleast_3d(X_new),
    np.atleast_3d(X_rec),
)  # (n, time, channels), also for 2-D input
shown_domains = np.unique(domains_new)[:4]
n_channels = inputs.shape[2]
fig, axes = plt.subplots(
    n_channels,
    len(shown_domains),
    figsize=(11, 2.3 * n_channels + 0.5),
    sharex=True,
    sharey="row",
    squeeze=False,
)
for col, domain in enumerate(shown_domains):
    row = np.flatnonzero(domains_new == domain)[0]
    for ch in range(n_channels):
        ax = axes[ch, col]
        ax.plot(inputs[row, :, ch], color="0.6", label="input")
        ax.plot(recons[row, :, ch], color="C0", label="RAVE reconstruction")
        if ch == 0:
            ax.set_title(f"domain {domain}")
        if col == 0:
            ax.set_ylabel(f"channel {ch}")
for ax in axes[-1]:
    ax.set_xlabel("time (samples)")
axes[0, 0].legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

for domain in np.unique(domains_new):
    rows = domains_new == domain
    print(f"domain {domain}: reconstruction MSE {np.mean((X_new[rows] - X_rec[rows]) ** 2):.3f}")
r2 = 1 - np.sum((X_new - X_rec) ** 2) / np.sum((X_new - X_new.mean(axis=0)) ** 2)
print(f"variance explained on new time series: {r2:.2f}")

## 4. RAVE+: partially known class labels

In the synthetic data, class labels are known in sessions `A` and `B` and **unknown in session `C`**, where they are marked with `None`. Every row is used for reconstruction and session removal, but only labeled rows train the class head.

In [ ]:
plus = RAVEPlus(latent_dim=8, domain_weight=1.0, seed=0).fit(X, domains=domains, y=y)

Z_plus = plus.transform(X_new)  # embedding, as with RAVE
X_rec_plus = plus.reconstruct(X_new)  # reconstruction, as with RAVE
y_pred = plus.predict(X_new)  # original label values
proba = plus.predict_proba(X_new)  # columns follow plus.classes_
print("classes_:", plus.classes_.tolist(), "| predict_proba:", proba.shape)

if y_new_true is not None:
    labeled_domains = {d for d, label in zip(domains, y, strict=True) if label is not None}
    for domain in np.unique(domains_new):
        rows = domains_new == domain
        note = (
            ""
            if domain in labeled_domains
            else "  <- no labels from this domain were used in training"
        )
        print(f"domain {domain}: accuracy {np.mean(y_pred[rows] == y_new_true[rows]):.2f}{note}")

## 5. Reading the results

These numbers come from synthetic data, where the session effect and the classes are separable by construction. On your own data, evaluate both sides of the trade-off before relying on the embedding:

- **Signal retention:** compare reconstruction error from `reconstruct()` on held-out time series, per domain, with a `domain_weight=0` fit. If known classes exist, check that they stay separable in the embedding across domains.
- **Domain removal:** the discriminator used during training is not a measurement. Train a separate classifier on held-out embeddings to see how much domain information remains, and compare against a `domain_weight=0` fit. Hold out whole recordings or sessions where you can.
- **Choosing `domain_weight`:** it trades removing domain information against keeping signal. Start from the default (0.1), try a few values (for example 0, 0.1 and 1.0) with several seeds, and compare both checks. If classes and domains are confounded (a class recorded mostly in one domain), removing the domain also removes class information.
- **Training curves:** `history_` holds per-epoch averages taken while the weights are still changing. The adversarial term has no upper bound, so these averages can swing a lot from epoch to epoch. If reconstruction error from `reconstruct()` approaches the variance of your data, lower `domain_weight` or `epochs`.

See the [README](../README.md) for the full input contract, the settings, and the differences from the original implementation.